# MASTER SDN DDoS DATASET GENERATOR

This notebook is the project's master dataset-generation framework.

It supports two modes:

1. **Synthetic mode** — generates statistically controlled SDN-flow observations without requiring Mininet.
2. **Real SDN lab mode** — documents and provides executable scaffolding for Mininet + Open vSwitch + Ryu/OpenFlow 1.3 telemetry and controlled traffic experiments.

The real-lab mode is deliberately restricted to an isolated Mininet laboratory. It is not designed to launch DDoS traffic against external systems or third-party amplification/reflection services.

## Final architecture

```text
Experiment Controller
        |
        +------------------+
        |                  |
        v                  v
   Mininet/OVS        Synthetic Engine
        |                  |
        +--------+---------+
                 |
          Flow/Port/Controller
             Telemetry
                 |
          Feature Extraction
                 |
          Window Aggregation
                 |
             Ground Truth
                 |
       +---------+----------+
       |                    |
      CSV                Parquet
       |
      PCAP metadata
       |
      Kafka later
       |
    PySpark later
```


## Attack/scenario catalogue

Core research-aligned classes:

- BENIGN_TCP
- BENIGN_UDP
- BENIGN_ICMP
- BENIGN_HTTP
- UDP_FLOOD
- ICMP_FLOOD
- TCP_SYN_FLOOD
- HTTP_FLOOD
- SLOW_RATE_DDOS
- UDP_AMPLIFICATION_STYLE
- SMURF_STYLE
- PING_OF_DEATH_STYLE
- LAND
- PACKET_IN_FLOOD

The `*_STYLE` scenarios are behavioral laboratory representations. They do not contact real reflection/amplification infrastructure or transmit malformed packets outside the isolated testbed.


In [1]:
from pathlib import Path
from dataclasses import dataclass, asdict
import json
import math
import platform
import subprocess
import shutil
import sys
import time

import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

ROOT = Path("sdn_ddos_dataset")
RAW = ROOT / "raw"
PROCESSED = ROOT / "processed"
METADATA = ROOT / "metadata"
SPLITS = ROOT / "splits"
REPORTS = ROOT / "reports"
PLOTS = REPORTS / "plots"

for p in [RAW, PROCESSED, METADATA, SPLITS, REPORTS, PLOTS]:
    p.mkdir(parents=True, exist_ok=True)

OBS_WINDOW_SECONDS = 5

ATTACK_TYPES = [
    "BENIGN_TCP",
    "BENIGN_UDP",
    "BENIGN_ICMP",
    "BENIGN_HTTP",
    "UDP_FLOOD",
    "ICMP_FLOOD",
    "TCP_SYN_FLOOD",
    "HTTP_FLOOD",
    "SLOW_RATE_DDOS",
    "UDP_AMPLIFICATION_STYLE",
    "SMURF_STYLE",
    "PING_OF_DEATH_STYLE",
    "LAND",
    "PACKET_IN_FLOOD",
]

LABEL_ID = {x: i for i, x in enumerate(ATTACK_TYPES)}

print("Platform:", platform.platform())
print("Python:", sys.version.split()[0])
print("Output:", ROOT.resolve())


Platform: Windows-11-10.0.26200-SP0
Python: 3.14.0
Output: D:\Bunker\BaseCamp\WirelessCommunication\DDoS\sdn_ddos_dataset


## 1. Environment checks

The synthetic mode runs on ordinary Python.

The real SDN mode expects a Linux environment with:

- Mininet
- Open vSwitch
- OpenFlow 1.3 support
- Ryu or OS-Ken
- tcpdump (optional)
- iperf3 (optional)

Mininet is normally easiest to operate in a Linux VM, native Linux machine, or suitable WSL/Linux setup. Windows-only execution is not assumed for the real-lab mode.


In [2]:
def command_exists(name):
    return shutil.which(name) is not None

required_real_lab_tools = ["mn", "ovs-vsctl", "tcpdump", "iperf3"]

tool_status = pd.DataFrame({
    "tool": required_real_lab_tools,
    "available": [command_exists(x) for x in required_real_lab_tools]
})

tool_status


,tool,available
0,mn,False
1,ovs-vsctl,False
2,tcpdump,False
3,iperf3,False


## 2. Experiment configuration

Every generated record carries scenario metadata.

This is important for leakage-aware evaluation: the final train/validation/test split should be performed by scenario, not by randomly splitting adjacent rows from the same experiment.


In [3]:
@dataclass
class Scenario:
    scenario_id: str
    attack_type: str
    topology: str
    attacker_count: int
    attack_intensity: str
    background_level: str
    duration_s: int
    seed: int

SCENARIO_LIBRARY = []

scenario_counter = 0

def add_scenario(attack_type, topology="linear_3switch",
                 attacker_count=1, intensity="medium",
                 background="medium", duration_s=60):
    global scenario_counter
    scenario_counter += 1
    s = Scenario(
        scenario_id=f"SCN_{scenario_counter:04d}",
        attack_type=attack_type,
        topology=topology,
        attacker_count=attacker_count,
        attack_intensity=intensity,
        background_level=background,
        duration_s=duration_s,
        seed=SEED + scenario_counter,
    )
    SCENARIO_LIBRARY.append(s)

# Core scenarios
for attack in ATTACK_TYPES:
    for intensity in ["low", "medium", "high"]:
        add_scenario(
            attack,
            topology="linear_3switch",
            attacker_count=1 if attack.startswith("BENIGN") else 2,
            intensity=intensity,
            background="medium",
            duration_s=60,
        )

scenario_df = pd.DataFrame([asdict(s) for s in SCENARIO_LIBRARY])
scenario_df.to_csv(METADATA / "scenarios.csv", index=False)

scenario_df.head(), scenario_df.shape


(  scenario_id attack_type        topology  attacker_count attack_intensity  \
 0    SCN_0001  BENIGN_TCP  linear_3switch               1              low   
 1    SCN_0002  BENIGN_TCP  linear_3switch               1           medium   
 2    SCN_0003  BENIGN_TCP  linear_3switch               1             high   
 3    SCN_0004  BENIGN_UDP  linear_3switch               1              low   
 4    SCN_0005  BENIGN_UDP  linear_3switch               1           medium   
 
   background_level  duration_s  seed  
 0           medium          60    43  
 1           medium          60    44  
 2           medium          60    45  
 3           medium          60    46  
 4           medium          60    47  ,
 (42, 8))

## 3. Statistical traffic model

The synthetic engine models the observable behavior of each scenario.

It generates correlated values rather than independently drawing every column.

For example:

`packet_rate = packet_count / observation_window`

and:

`byte_rate = byte_count / observation_window`

TCP scenarios additionally model SYN/ACK/RST behavior. SDN-oriented scenarios model flow-table growth and controller-load indicators.

These are synthetic proxies. Real-lab telemetry remains the preferred source for final experimental claims.


In [4]:
INTENSITY_MULTIPLIER = {
    "low": 0.55,
    "medium": 1.0,
    "high": 1.8,
}

BACKGROUND_MULTIPLIER = {
    "low": 0.65,
    "medium": 1.0,
    "high": 1.45,
}

def positive_lognormal(median, sigma=0.35, multiplier=1.0):
    return max(0.01, float(rng.lognormal(np.log(median * multiplier), sigma)))

def bounded_normal(mean, std, low, high):
    return float(np.clip(rng.normal(mean, std), low, high))

def beta(a, b):
    return float(rng.beta(a, b))

def scenario_behavior(s: Scenario):
    mult = INTENSITY_MULTIPLIER[s.attack_intensity]
    bg = BACKGROUND_MULTIPLIER[s.background_level]
    a = s.attack_type

    if a.startswith("BENIGN"):
        base = {
            "packet_rate": positive_lognormal(100 * bg, .40),
            "avg_packet_size": bounded_normal(700, 220, 80, 1500),
            "new_flows": positive_lognormal(5 * bg, .40),
            "active_flows": positive_lognormal(20 * bg, .30),
            "completion": beta(8, 2),
            "request_rate": positive_lognormal(8 * bg, .40),
            "source_entropy": bounded_normal(.82, .08, .25, 1),
            "table_growth": positive_lognormal(2 * bg, .35),
            "controller_load": bounded_normal(.12 * bg, .04, .01, .35),
            "syn": positive_lognormal(8, .35),
            "ack": positive_lognormal(12, .35),
            "rst": positive_lognormal(1.5, .45),
        }
        if a == "BENIGN_HTTP":
            base["request_rate"] *= 1.5
            base["active_flows"] *= 1.3
            base["controller_load"] += .04
        if a == "BENIGN_ICMP":
            base["avg_packet_size"] *= .45
        if a == "BENIGN_UDP":
            base["avg_packet_size"] *= .65
        return base

    if a == "UDP_FLOOD":
        return dict(
            packet_rate=positive_lognormal(1800, .45, mult),
            avg_packet_size=bounded_normal(420, 100, 64, 900),
            new_flows=positive_lognormal(90, .45, mult),
            active_flows=positive_lognormal(250, .45, mult),
            completion=beta(2, 8),
            request_rate=positive_lognormal(320, .45, mult),
            source_entropy=bounded_normal(.38, .12, .03, .8),
            table_growth=positive_lognormal(45, .45, mult),
            controller_load=bounded_normal(.65 * min(mult, 1.8), .10, .1, 1),
            syn=0, ack=0, rst=0,
        )

    if a == "ICMP_FLOOD":
        return dict(
            packet_rate=positive_lognormal(2200, .40, mult),
            avg_packet_size=bounded_normal(256, 55, 64, 700),
            new_flows=positive_lognormal(65, .45, mult),
            active_flows=positive_lognormal(180, .40, mult),
            completion=beta(1.5, 9),
            request_rate=positive_lognormal(380, .40, mult),
            source_entropy=bounded_normal(.30, .10, .02, .7),
            table_growth=positive_lognormal(30, .45, mult),
            controller_load=bounded_normal(.55 * min(mult, 1.8), .10, .1, 1),
            syn=0, ack=0, rst=0,
        )

    if a == "TCP_SYN_FLOOD":
        return dict(
            packet_rate=positive_lognormal(1400, .50, mult),
            avg_packet_size=bounded_normal(64, 10, 40, 120),
            new_flows=positive_lognormal(220, .50, mult),
            active_flows=positive_lognormal(500, .45, mult),
            completion=beta(1, 20),
            request_rate=positive_lognormal(300, .45, mult),
            source_entropy=bounded_normal(.42, .12, .03, .8),
            table_growth=positive_lognormal(100, .45, mult),
            controller_load=bounded_normal(.85, .08, .3, 1),
            syn=positive_lognormal(900, .40, mult),
            ack=positive_lognormal(8, .50),
            rst=positive_lognormal(55, .50, mult),
        )

    if a == "HTTP_FLOOD":
        return dict(
            packet_rate=positive_lognormal(700, .45, mult),
            avg_packet_size=bounded_normal(850, 220, 150, 1500),
            new_flows=positive_lognormal(80, .45, mult),
            active_flows=positive_lognormal(220, .40, mult),
            completion=beta(7, 3),
            request_rate=positive_lognormal(180, .40, mult),
            source_entropy=bounded_normal(.68, .10, .1, 1),
            table_growth=positive_lognormal(55, .40, mult),
            controller_load=bounded_normal(.62, .12, .15, 1),
            syn=positive_lognormal(120, .40, mult),
            ack=positive_lognormal(180, .40, mult),
            rst=positive_lognormal(10, .50),
        )

    if a == "SLOW_RATE_DDOS":
        return dict(
            packet_rate=bounded_normal(25 * mult, 10, 4, 180),
            avg_packet_size=bounded_normal(120, 30, 40, 300),
            new_flows=positive_lognormal(130, .40, mult),
            active_flows=positive_lognormal(420, .40, mult),
            completion=beta(1, 12),
            request_rate=bounded_normal(35 * mult, 12, 2, 150),
            source_entropy=bounded_normal(.72, .10, .1, 1),
            table_growth=positive_lognormal(80, .40, mult),
            controller_load=bounded_normal(.78, .10, .2, 1),
            syn=positive_lognormal(100, .40, mult),
            ack=bounded_normal(18, 8, 0, 50),
            rst=bounded_normal(2, 2, 0, 15),
        )

    if a == "UDP_AMPLIFICATION_STYLE":
        return dict(
            packet_rate=positive_lognormal(3500, .40, mult),
            avg_packet_size=bounded_normal(950, 220, 150, 1500),
            new_flows=positive_lognormal(35, .40, mult),
            active_flows=positive_lognormal(120, .40, mult),
            completion=beta(1.5, 10),
            request_rate=positive_lognormal(260, .35, mult),
            source_entropy=bounded_normal(.52, .12, .05, .9),
            table_growth=positive_lognormal(20, .40, mult),
            controller_load=bounded_normal(.72, .10, .2, 1),
            syn=0, ack=0, rst=0,
        )

    if a == "SMURF_STYLE":
        return dict(
            packet_rate=positive_lognormal(2600, .50, mult),
            avg_packet_size=bounded_normal(500, 130, 80, 1200),
            new_flows=positive_lognormal(100, .50, mult),
            active_flows=positive_lognormal(300, .45, mult),
            completion=beta(1.5, 9),
            request_rate=positive_lognormal(450, .45, mult),
            source_entropy=bounded_normal(.25, .10, .02, .65),
            table_growth=positive_lognormal(40, .40, mult),
            controller_load=bounded_normal(.68, .12, .2, 1),
            syn=0, ack=0, rst=0,
        )

    if a == "PING_OF_DEATH_STYLE":
        return dict(
            packet_rate=positive_lognormal(900, .45, mult),
            avg_packet_size=bounded_normal(1450, 60, 1200, 1500),
            new_flows=positive_lognormal(20, .40, mult),
            active_flows=positive_lognormal(80, .35, mult),
            completion=beta(2, 7),
            request_rate=positive_lognormal(140, .40, mult),
            source_entropy=bounded_normal(.55, .12, .05, .9),
            table_growth=positive_lognormal(12, .35, mult),
            controller_load=bounded_normal(.42, .10, .1, 1),
            syn=0, ack=0, rst=0,
        )

    if a == "LAND":
        return dict(
            packet_rate=positive_lognormal(900, .50, mult),
            avg_packet_size=bounded_normal(80, 20, 40, 200),
            new_flows=positive_lognormal(100, .45, mult),
            active_flows=positive_lognormal(160, .40, mult),
            completion=beta(1, 12),
            request_rate=positive_lognormal(180, .45, mult),
            source_entropy=bounded_normal(.02, .02, .0, .15),
            table_growth=positive_lognormal(50, .40, mult),
            controller_load=bounded_normal(.60, .12, .1, 1),
            syn=positive_lognormal(300, .45, mult),
            ack=positive_lognormal(3, .5),
            rst=positive_lognormal(40, .5, mult),
        )

    if a == "PACKET_IN_FLOOD":
        return dict(
            packet_rate=positive_lognormal(500, .50, mult),
            avg_packet_size=bounded_normal(120, 35, 40, 400),
            new_flows=positive_lognormal(700, .50, mult),
            active_flows=positive_lognormal(800, .45, mult),
            completion=beta(1, 15),
            request_rate=positive_lognormal(700, .50, mult),
            source_entropy=bounded_normal(.35, .12, .03, .8),
            table_growth=positive_lognormal(350, .45, mult),
            controller_load=bounded_normal(.92, .06, .45, 1),
            syn=positive_lognormal(80, .5, mult),
            ack=positive_lognormal(3, .5),
            rst=positive_lognormal(20, .5, mult),
        )

    raise ValueError(a)


In [5]:
def random_ip():
    return f"10.0.0.{int(rng.integers(1, 49))}"

def random_port():
    return int(rng.integers(1024, 65535))

def protocol_for(attack):
    if "ICMP" in attack or "PING" in attack or "SMURF" in attack:
        return "ICMP"
    if "UDP" in attack:
        return "UDP"
    return "TCP"

def generate_observation(s: Scenario, timestamp):
    b = scenario_behavior(s)
    protocol = protocol_for(s.attack_type)

    if s.attack_type.startswith("BENIGN"):
        if s.attack_type == "BENIGN_HTTP":
            dst_port = int(rng.choice([80, 443]))
        elif s.attack_type == "BENIGN_TCP":
            dst_port = int(rng.choice([22, 80, 443, 8080]))
        else:
            dst_port = random_port()
    else:
        dst_port = random_port()

    src_ip = random_ip()
    dst_ip = random_ip()

    # Preserve a small probability of repeated source/destination pairs.
    if s.attack_type == "LAND":
        dst_ip = src_ip

    packets = max(1, int(round(b["packet_rate"] * OBS_WINDOW_SECONDS)))
    avg_size = b["avg_packet_size"]
    bytes_ = max(packets, packets * avg_size)

    new_flows = max(1, int(round(b["new_flows"])))
    active_flows = max(new_flows, int(round(b["active_flows"])))

    return {
        "scenario_id": s.scenario_id,
        "experiment_id": f"EXP_{s.scenario_id}",
        "timestamp": timestamp,
        "window_start": timestamp,
        "window_end": timestamp + pd.Timedelta(seconds=OBS_WINDOW_SECONDS),
        "window_duration_s": OBS_WINDOW_SECONDS,

        "topology": s.topology,
        "switch_id": f"s{int(rng.integers(1, 4))}",
        "input_port": int(rng.integers(1, 5)),
        "output_port": int(rng.integers(1, 5)),

        "src_ip": src_ip,
        "dst_ip": dst_ip,
        "src_port": random_port(),
        "dst_port": dst_port,
        "protocol": protocol,

        "packet_count": packets,
        "byte_count": round(bytes_, 2),
        "packet_rate": round(b["packet_rate"], 4),
        "byte_rate": round(bytes_ / OBS_WINDOW_SECONDS, 4),
        "avg_packet_size": round(avg_size, 4),

        "flow_duration_s": round(bounded_normal(
            4.0 if not s.attack_type.startswith("BENIGN") else 2.8,
            0.8, .2, OBS_WINDOW_SECONDS
        ), 4),

        "new_flows": new_flows,
        "active_flows": active_flows,
        "flow_rate": round(new_flows / OBS_WINDOW_SECONDS, 4),
        "flow_churn": round(new_flows / max(active_flows, 1), 5),

        "tcp_syn_count": int(max(0, b["syn"])),
        "tcp_ack_count": int(max(0, b["ack"])),
        "tcp_rst_count": int(max(0, b["rst"])),
        "tcp_fin_count": int(max(0, b["ack"] * .2 if protocol == "TCP" else 0)),

        "connection_attempts": int(max(1, b["new_flows"])),
        "completed_connections": int(max(0, b["new_flows"] * b["completion"])),
        "tcp_connection_completion_ratio": round(b["completion"], 5),

        "request_count": int(max(1, b["request_rate"] * OBS_WINDOW_SECONDS)),
        "request_rate": round(b["request_rate"], 4),

        "source_entropy": round(b["source_entropy"], 5),
        "destination_entropy": round(
            bounded_normal(.75 if s.attack_type.startswith("BENIGN") else .35, .12, .02, 1), 5
        ),
        "port_entropy": round(
            bounded_normal(.70 if s.attack_type.startswith("BENIGN") else .30, .14, .02, 1), 5
        ),

        "in_packets": packets,
        "out_packets": int(max(1, packets * bounded_normal(.8, .2, .05, 1.5))),
        "in_bytes": round(bytes_, 2),
        "out_bytes": round(max(1, bytes_ * bounded_normal(.8, .2, .05, 1.5)), 2),

        "flow_table_entries": int(max(1, b["active_flows"] * bounded_normal(1.0, .12, .5, 1.5))),
        "flow_table_growth": round(b["table_growth"], 4),

        "packet_in_count": int(max(0, b["new_flows"] * bounded_normal(
            0.3 if not s.attack_type == "PACKET_IN_FLOOD" else 1.8, .25, .0, 3.0
        ))),
        "flow_mod_count": int(max(0, b["new_flows"] * bounded_normal(.2, .15, .0, 2.0))),
        "controller_load_indicator": round(b["controller_load"], 5),

        "attacker_count": s.attacker_count,
        "attack_intensity": s.attack_intensity,
        "background_traffic_level": s.background_level,

        "attack_type": s.attack_type,
        "label": 0 if s.attack_type.startswith("BENIGN") else 1,
        "label_id": 0 if s.attack_type.startswith("BENIGN") else LABEL_ID[s.attack_type],
    }


## 4. Generate the synthetic master dataset

This produces a broad development dataset from the scenario library.

For the first run, use a moderate number of observations. Once the schema and ML pipeline are validated, scale `WINDOWS_PER_SCENARIO`.


In [6]:
WINDOWS_PER_SCENARIO = 300

rows = []
base_time = pd.Timestamp("2026-01-01 00:00:00", tz="UTC")

for scenario_index, s in enumerate(SCENARIO_LIBRARY):
    scenario_start = base_time + pd.Timedelta(days=scenario_index)
    for w in range(WINDOWS_PER_SCENARIO):
        ts = scenario_start + pd.Timedelta(seconds=w * OBS_WINDOW_SECONDS)
        rows.append(generate_observation(s, ts))

master_df = pd.DataFrame(rows)

# Shuffle for downstream ML experiments, while preserving scenario_id.
master_df = master_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Rows:", len(master_df))
print("Columns:", len(master_df.columns))
print("\nAttack distribution:")
display(master_df["attack_type"].value_counts())

master_df.head()


Rows: 12600
Columns: 52

Attack distribution:


attack_type
LAND                       900
HTTP_FLOOD                 900
BENIGN_UDP                 900
BENIGN_ICMP                900
UDP_FLOOD                  900
BENIGN_HTTP                900
BENIGN_TCP                 900
SLOW_RATE_DDOS             900
PACKET_IN_FLOOD            900
PING_OF_DEATH_STYLE        900
UDP_AMPLIFICATION_STYLE    900
TCP_SYN_FLOOD              900
ICMP_FLOOD                 900
SMURF_STYLE                900
Name: count, dtype: int64

,scenario_id,experiment_id,timestamp,window_start,window_end,window_duration_s,topology,switch_id,input_port,output_port,...,flow_table_growth,packet_in_count,flow_mod_count,controller_load_indicator,attacker_count,attack_intensity,background_traffic_level,attack_type,label,label_id
0,SCN_0038,EXP_SCN_0038,2026-02-07 00:23:45+00:00,2026-02-07 00:23:45+00:00,2026-02-07 00:23:50+00:00,5,linear_3switch,s1,4,4,...,43.5980,6,0,0.62048,2,medium,medium,LAND,1,12
1,SCN_0023,EXP_SCN_0023,2026-01-23 00:22:10+00:00,2026-01-23 00:22:10+00:00,2026-01-23 00:22:15+00:00,5,linear_3switch,s3,2,2,...,36.9536,14,3,0.55853,2,medium,medium,HTTP_FLOOD,1,7
2,SCN_0005,EXP_SCN_0005,2026-01-05 00:10:15+00:00,2026-01-05 00:10:15+00:00,2026-01-05 00:10:20+00:00,5,linear_3switch,s1,1,2,...,1.3194,1,1,0.15187,1,medium,medium,BENIGN_UDP,0,0
3,SCN_0008,EXP_SCN_0008,2026-01-08 00:12:40+00:00,2026-01-08 00:12:40+00:00,2026-01-08 00:12:45+00:00,5,linear_3switch,s2,1,2,...,1.7657,1,1,0.09926,1,medium,medium,BENIGN_ICMP,0,0
4,SCN_0038,EXP_SCN_0038,2026-02-07 00:14:45+00:00,2026-02-07 00:14:45+00:00,2026-02-07 00:14:50+00:00,5,linear_3switch,s2,1,3,...,23.5296,51,19,0.63786,2,medium,medium,LAND,1,12


## 5. Dataset validation

We check:

- missing values
- duplicate observations
- impossible rates
- invalid packet/byte relationships
- label consistency
- class distribution
- scenario coverage


In [7]:
def validate_dataset(df):
    checks = []

    checks.append(("missing_cells", int(df.isna().sum().sum())))
    checks.append(("duplicate_rows", int(df.duplicated().sum())))
    checks.append(("negative_packet_count", int((df.packet_count < 0).sum())))
    checks.append(("negative_byte_count", int((df.byte_count < 0).sum())))
    checks.append(("negative_packet_rate", int((df.packet_rate < 0).sum())))
    checks.append(("byte_rate_mismatch", int(
        ~np.isclose(
            df["byte_rate"].to_numpy(),
            (df["byte_count"] / df["window_duration_s"]).to_numpy(),
            rtol=1e-4, atol=1e-3
        )
    ).sum()))
    checks.append(("invalid_labels", int((~df["label"].isin([0, 1])).sum())))
    checks.append(("scenario_count", int(df["scenario_id"].nunique())))

    return pd.DataFrame(checks, columns=["check", "value"])

validation = validate_dataset(master_df)
validation


TypeError: only 0-dimensional arrays can be converted to Python scalars

## 6. Feature dictionary

The feature dictionary is exported so that the final project report can explicitly explain what every column means and where it originates.

In real SDN mode, some features are direct OVS/Ryu measurements and others are derived from those measurements.


In [ ]:
feature_dictionary = pd.DataFrame([
    ("packet_count", "Flow/window", "Number of observed packets"),
    ("byte_count", "Flow/window", "Number of observed bytes"),
    ("packet_rate", "Derived", "Packets per observation second"),
    ("byte_rate", "Derived", "Bytes per observation second"),
    ("avg_packet_size", "Derived", "Byte count divided by packet count"),
    ("new_flows", "SDN/flow", "New flow observations in the window"),
    ("active_flows", "SDN/flow", "Estimated active flows"),
    ("flow_rate", "Derived", "New flows per second"),
    ("flow_churn", "Derived", "New flows divided by active flows"),
    ("tcp_syn_count", "TCP", "Observed SYN-related count"),
    ("tcp_ack_count", "TCP", "Observed ACK-related count"),
    ("tcp_rst_count", "TCP", "Observed RST-related count"),
    ("tcp_fin_count", "TCP", "Observed FIN-related count"),
    ("tcp_connection_completion_ratio", "Derived", "Completed connections divided by attempts"),
    ("request_rate", "Application", "Application request rate"),
    ("source_entropy", "Derived", "Source-distribution entropy proxy"),
    ("destination_entropy", "Derived", "Destination-distribution entropy proxy"),
    ("port_entropy", "Derived", "Destination/source port entropy proxy"),
    ("flow_table_entries", "SDN", "Observed/estimated flow-table occupancy"),
    ("flow_table_growth", "SDN", "Flow-table growth during the window"),
    ("packet_in_count", "SDN control plane", "Packet-In event count"),
    ("flow_mod_count", "SDN control plane", "Flow modification count"),
    ("controller_load_indicator", "SDN control plane", "Normalized controller-pressure indicator"),
    ("attack_type", "Ground truth", "Scenario class"),
    ("label", "Ground truth", "0 benign, 1 attack"),
], columns=["feature", "category", "definition"])

feature_dictionary.to_csv(METADATA / "feature_dictionary.csv", index=False)
feature_dictionary


## 7. Scenario-aware train/validation/test split

Do not randomly split individual rows from the same scenario.

We split by `scenario_id`, ensuring that a complete scenario belongs to only one partition.


In [ ]:
scenario_ids = master_df["scenario_id"].drop_duplicates().to_numpy()
rng_split = np.random.default_rng(SEED)
rng_split.shuffle(scenario_ids)

n = len(scenario_ids)
train_ids = set(scenario_ids[:int(.70*n)])
val_ids = set(scenario_ids[int(.70*n):int(.85*n)])
test_ids = set(scenario_ids[int(.85*n):])

train_df = master_df[master_df.scenario_id.isin(train_ids)].copy()
val_df = master_df[master_df.scenario_id.isin(val_ids)].copy()
test_df = master_df[master_df.scenario_id.isin(test_ids)].copy()

print("Train scenarios:", len(train_ids), "rows:", len(train_df))
print("Validation scenarios:", len(val_ids), "rows:", len(val_df))
print("Test scenarios:", len(test_ids), "rows:", len(test_df))

assert not train_ids & val_ids
assert not train_ids & test_ids
assert not val_ids & test_ids


## 8. Export datasets

Parquet is the preferred format for later Spark processing. CSV remains useful for inspection and interoperability.


In [ ]:
master_csv = PROCESSED / "master_sdn_ddos_dataset.csv"
master_parquet = PROCESSED / "master_sdn_ddos_dataset.parquet"

train_parquet = SPLITS / "train.parquet"
val_parquet = SPLITS / "validation.parquet"
test_parquet = SPLITS / "test.parquet"

master_df.to_csv(master_csv, index=False)

for frame, path in [
    (master_df, master_parquet),
    (train_df, train_parquet),
    (val_df, val_parquet),
    (test_df, test_parquet),
]:
    try:
        frame.to_parquet(path, index=False)
    except Exception as e:
        print("Parquet unavailable:", e)
        break

print("CSV:", master_csv)
print("Parquet:", master_parquet)


## 9. Statistical inspection

These plots are intended to answer a basic question before ML:

> Do the classes actually exhibit different observable behavior?

A useful dataset should have overlap between some classes while retaining meaningful behavioral differences.


In [ ]:
import matplotlib.pyplot as plt

plot_features = [
    "packet_rate",
    "byte_rate",
    "new_flows",
    "active_flows",
    "flow_churn",
    "tcp_connection_completion_ratio",
    "packet_in_count",
    "controller_load_indicator",
]

for feature in plot_features:
    plt.figure(figsize=(12, 5))
    for label, group in master_df.groupby("attack_type"):
        plt.hist(group[feature], bins=40, alpha=0.30, label=label)
    plt.title(feature)
    plt.xlabel(feature)
    plt.ylabel("Frequency")
    plt.legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(PLOTS / f"{feature}.png", dpi=150)
    plt.show()


# REAL SDN LAB MODE

The following section is the bridge from synthetic data to measured data.

The intended real architecture is:

```text
Mininet
  |
  +-- OVS S1 -- OVS S2 -- OVS S3
  |
  +-- hosts / server / controlled attackers
  |
OpenFlow 1.3
  |
Ryu controller
  |
flow statistics + port statistics + Packet-In telemetry
  |
CSV/Parquet
```

The real mode must run in an isolated laboratory. The attack generators should target only Mininet virtual hosts.

The controller should periodically collect flow and port statistics and write records using the same schema as the synthetic engine.


In [ ]:
REAL_LAB_CONFIG = {
    "controller_ip": "127.0.0.1",
    "controller_port": 6633,
    "openflow_version": "1.3",
    "topology": "linear_3switch",
    "hosts": 12,
    "switches": 3,
    "victim_host": "h12",
    "observation_interval_s": 5,
    "isolate_to_mininet": True,
}

print(json.dumps(REAL_LAB_CONFIG, indent=2))


## 10. Mininet topology definition

Conceptual topology:

```text
                    Ryu
                     |
               OpenFlow 1.3
                     |
        h1 h2 h3     S1
                     |
                     S2
                     |
                     S3
                  /  |  \
                h10 h11 h12
                         |
                       victim
```

The helper below is a reference topology definition intended to be saved as a `.py` file and executed in the Mininet environment.

It is not executed by this notebook's Python kernel because Mininet's network namespaces and OVS services normally require a Linux environment with root privileges.


In [ ]:
MININET_TOPOLOGY = r'''
from mininet.topo import Topo

class SDNDDOSTopo(Topo):
    def build(self):
        switches = [self.addSwitch(f"s{i}") for i in range(1, 4)]
        hosts = [self.addHost(f"h{i}", ip=f"10.0.0.{i}/24") for i in range(1, 13)]

        # Host distribution
        for i, h in enumerate(hosts[:4]):
            self.addLink(h, switches[0])
        for i, h in enumerate(hosts[4:8]):
            self.addLink(h, switches[1])
        for i, h in enumerate(hosts[8:]):
            self.addLink(h, switches[2])

        self.addLink(switches[0], switches[1])
        self.addLink(switches[1], switches[2])

topos = {"sdnddos": lambda: SDNDDOSTopo()}
'''

topology_path = METADATA / "sdn_ddos_topology.py"
topology_path.write_text(MININET_TOPOLOGY, encoding="utf-8")
print(topology_path)


## 11. Ryu/OpenFlow telemetry collector

The collector below is the core of the real dataset path.

It requests:

- flow statistics
- port statistics
- Packet-In counts
- switch identifiers
- OpenFlow match information
- packet and byte counters

The resulting telemetry should be mapped into the same canonical dataset schema used above.

Run this portion in a Linux environment where Ryu is installed.

The code intentionally contains no external-target attack automation.


In [ ]:
RYU_COLLECTOR = r'''
from ryu.base import app_manager
from ryu.controller import ofp_event
from ryu.controller.handler import CONFIG_DISPATCHER, MAIN_DISPATCHER, set_ev_cls
from ryu.ofproto import ofproto_v1_3
from ryu.lib import hub
import csv
import time

class FlowCollector(app_manager.RyuApp):
    OFP_VERSIONS = [ofproto_v1_3.OFP_VERSION]

    def __init__(self, *args, **kwargs):
        super(FlowCollector, self).__init__(*args, **kwargs)
        self.datapaths = {}
        self.csv_path = "flow_statistics.csv"
        self.monitor_thread = hub.spawn(self._monitor)

    @set_ev_cls(ofp_event.EventOFPStateChange,
                [CONFIG_DISPATCHER, MAIN_DISPATCHER])
    def state_change_handler(self, ev):
        datapath = ev.datapath
        self.datapaths[datapath.id] = datapath

    def _monitor(self):
        while True:
            for datapath in list(self.datapaths.values()):
                self._request_stats(datapath)
            hub.sleep(5)

    def _request_stats(self, datapath):
        parser = datapath.ofproto_parser
        req = parser.OFPFlowStatsRequest(datapath)
        datapath.send_msg(req)

        port_req = parser.OFPPortStatsRequest(
            datapath, 0, datapath.ofproto.OFPP_ANY
        )
        datapath.send_msg(port_req)

    @set_ev_cls(ofp_event.EventOFPFlowStatsReply, MAIN_DISPATCHER)
    def flow_stats_reply_handler(self, ev):
        timestamp = time.time()
        datapath = ev.msg.datapath

        for stat in ev.msg.body:
            self.logger.info(
                "FLOW ts=%s dpid=%s packets=%s bytes=%s duration=%s match=%s",
                timestamp,
                datapath.id,
                stat.packet_count,
                stat.byte_count,
                stat.duration_sec,
                stat.match
            )
'''

collector_path = METADATA / "ryu_flow_collector.py"
collector_path.write_text(RYU_COLLECTOR, encoding="utf-8")
print(collector_path)


## 12. Controlled traffic scenarios

The real-lab traffic runner should be implemented around Mininet host namespaces.

Recommended benign traffic:

- ping
- iperf3 TCP
- iperf3 UDP
- HTTP requests to a local Mininet web server

Recommended controlled attack scenarios:

- SYN-flood behavior against a Mininet victim
- UDP flood against a Mininet victim
- ICMP flood against a Mininet victim
- HTTP request flood against a local Mininet web server
- slow-rate HTTP connection behavior against the local web server

For amplification/Smurf/Ping-of-Death/LAND, use **behavioral or synthetic representations** unless the experiment is specifically implemented and validated inside the isolated virtual topology. Do not point the generator at external infrastructure.

The dataset label must be synchronized to the scenario timeline.


## 13. Ground-truth timeline

A real experiment should produce a machine-readable timeline such as:

```text
scenario_id, start, end, attack_type
SCN_0001, 00:00, 00:30, BENIGN_TCP
SCN_0001, 00:30, 01:00, TCP_SYN_FLOOD
SCN_0001, 01:00, 01:30, BENIGN_TCP
```

Flow observations are labeled by overlap with this timeline.

This is substantially safer and more reproducible than manually labeling packets after the experiment.


In [ ]:
def label_by_timeline(flow_df, timeline_df):
    flow_df = flow_df.copy()
    flow_df["label"] = 0
    flow_df["attack_type"] = "BENIGN"

    for _, event in timeline_df.iterrows():
        mask = (
            (flow_df["timestamp"] >= event["start"]) &
            (flow_df["timestamp"] < event["end"])
        )
        flow_df.loc[mask, "attack_type"] = event["attack_type"]
        flow_df.loc[mask, "label"] = 0 if event["attack_type"].startswith("BENIGN") else 1

    return flow_df


## 14. Final data contract

The synthetic and real SDN pipelines must eventually produce the same canonical schema.

That gives us a clean architecture:

```text
                 +----------------+
                 | Synthetic mode |
                 +-------+--------+
                         |
                         |
                         v
                    CANONICAL
                  FLOW EVENT SCHEMA
                         ^
                         |
                         |
                 +-------+--------+
                 | Real SDN mode  |
                 +----------------+

                         |
                         v
                      Kafka
                         |
                         v
                     PySpark
                         |
                         v
                       ML
```

This is the key architectural decision for the project.

We can develop the Big Data pipeline now using synthetic records, then replace the producer with real Ryu/OVS telemetry later.


# 15. Recommended project progression

### Phase A — Dataset development

1. Run this notebook in synthetic mode.
2. Inspect distributions.
3. Add scenario variation.
4. Establish the canonical schema.
5. Export Parquet.

### Phase B — SDN laboratory

1. Install Mininet.
2. Install OVS.
3. Install Ryu/OS-Ken.
4. Run the 3-switch topology.
5. Verify OpenFlow 1.3.
6. Run the telemetry collector.
7. Generate controlled local traffic.
8. Validate measured flow statistics.

### Phase C — Real dataset

1. Synchronize scenario timeline.
2. Collect flow statistics.
3. Collect port statistics.
4. Collect controller events.
5. Optionally capture PCAP.
6. Convert to canonical schema.
7. Label automatically.
8. Export Parquet.

### Phase D — Big Data

```text
Ryu
 ↓
Kafka
 ↓
PySpark Structured Streaming
 ↓
window aggregation
 ↓
feature engineering
 ↓
ML
```

### Phase E — Detection and mitigation

```text
ML
 ↓
attack probability
 ↓
Ryu
 ↓
OpenFlow rule
 ↓
OVS
 ↓
drop / rate-limit / isolate
```
